In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import to_scipy_sparse_matrix
import scipy.sparse as sp
import numpy as np

# Load the saved graph data
data_path = './graph_data/all_graphs_data.pt'
graphs_data = torch.load(data_path, weights_only=False)

print("Loaded graphs:", graphs_data.keys())

c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded graphs: dict_keys(['StandardScaled_Distance_Graph', 'DTW_Graph', 'CID_Graph'])


In [ ]:
from GNN.GNNs.GCN import GCN

# Util function to convert edge_index to sparse torch adjacency matrix
def sparse_adj_from_edge_index(edge_index, edge_attr, num_nodes):
    # Convert to scipy sparse matrix using PyG utility
    adj = to_scipy_sparse_matrix(edge_index, edge_attr=edge_attr, num_nodes=num_nodes)
    adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj) # Ensure symmetric
    
    # Normalize adjacency matrix (D^-0.5 A_hat D^-0.5)
    adj = adj + sp.eye(adj.shape[0])  # Add self-loops
    rowsum = np.array(adj.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    adj_norm = adj.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)
    
    # Convert back to torch sparse tensor
    adj_coo = adj_norm.tocoo()
    indices = torch.from_numpy(np.vstack((adj_coo.row, adj_coo.col)).astype(np.int64))
    values = torch.from_numpy(adj_coo.data.astype(np.float32))
    size = torch.Size(adj_coo.shape)
    
    return torch.sparse_coo_tensor(indices, values, size)

In [ ]:
# Define hyperparams
NHID = 16 # Hidden layer size
NCLASS = 5 # arbitrary classes since we're generating embeddings
DROPOUT = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
graph_to_be_used = 'CID_Graph_data.pt' # Change this to select which graph's embeddings to use

pyg_data = torch.load(f"./graph_data/{graph_to_be_used}", weights_only=False)
graph_name = graph_to_be_used.replace(".pt", "")
print(f"\nProcessing {graph_name}...")
x = pyg_data.x.to(DEVICE)
edge_index = pyg_data.edge_index.to(DEVICE)
    
# Calculate sparse Adjacency matrix for Custom GCN
adj = sparse_adj_from_edge_index(edge_index, None, x.size(0)) # Pass edge_attr=None for unweighted
adj = adj.to(DEVICE)
    
# Setup GCN layer structure and init
nfeat = x.shape[1]
model = GCN(nfeat=nfeat, nhid=NHID, nclass=NCLASS, dropout=DROPOUT).to(DEVICE)
model.eval()
    


print("\nFinished generating embeddings for all invariances.")


Processing CID_Graph_data...

Finished generating embeddings for all invariances.
